In [65]:
import pandas as pd
import numpy as np

In [66]:
df = pd.read_csv("../data/raw/customer_support_tickets.csv")

In [67]:
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")

In [68]:
df.head()

,ticket_id,customer_name,customer_email,customer_age,customer_gender,product_purchased,date_of_purchase,ticket_type,ticket_subject,ticket_description,ticket_status,resolution,ticket_priority,ticket_channel,first_response_time,time_to_resolution,customer_satisfaction_rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [69]:
print("Shape:", df.shape)
print("\nMissing Values:\n", df.isnull().sum())
print("\nData Types:\n", df.dtypes)

Shape: (8469, 17)

Missing Values:
 ticket_id                          0
customer_name                      0
customer_email                     0
customer_age                       0
customer_gender                    0
product_purchased                  0
date_of_purchase                   0
ticket_type                        0
ticket_subject                     0
ticket_description                 0
ticket_status                      0
resolution                      5700
ticket_priority                    0
ticket_channel                     0
first_response_time             2819
time_to_resolution              5700
customer_satisfaction_rating    5700
dtype: int64

Data Types:
 ticket_id                         int64
customer_name                    object
customer_email                   object
customer_age                      int64
customer_gender                  object
product_purchased                object
date_of_purchase                 object
ticket_type                 

In [70]:
df["date_of_purchase"] = pd.to_datetime(df["date_of_purchase"], errors="coerce")
df["first_response_time"] = pd.to_datetime(df["first_response_time"], errors="coerce")
df["time_to_resolution"] = pd.to_datetime(df["time_to_resolution"], errors="coerce")

In [71]:
df["is_closed"] = df["ticket_status"].str.lower() == "closed"
df["is_open"] = ~df["is_closed"]

In [72]:
# Resolution time (ONLY for valid rows)
df["resolution_time_hours"] = np.where(
    df["time_to_resolution"].notna() & df["first_response_time"].notna(),
    (df["time_to_resolution"] - df["first_response_time"]).dt.total_seconds() / 3600,
    np.nan
)

# Remove negative values (data errors)
df.loc[df["resolution_time_hours"] < 0, "resolution_time_hours"] = np.nan

In [73]:
# Approximate response delay (relative to purchase date)
df["response_delay_hours"] = np.where(
    df["first_response_time"].notna() & df["date_of_purchase"].notna(),
    (df["first_response_time"] - df["date_of_purchase"]).dt.total_seconds() / 3600,
    np.nan
)

df.loc[df["response_delay_hours"] < 0, "response_delay_hours"] = np.nan

In [74]:
analysis_df = df[
    (df["resolution_time_hours"].notna()) &
    (df["customer_satisfaction_rating"].notna())
].copy()

print("Analysis dataset shape:", analysis_df.shape)

Analysis dataset shape: (1404, 21)


In [75]:
threshold = analysis_df["resolution_time_hours"].quantile(0.75)

analysis_df["is_delayed"] = analysis_df["resolution_time_hours"] > threshold

In [76]:
analysis_df["customer_effort_score"] = (
    analysis_df["resolution_time_hours"] / 10 +
    analysis_df["is_delayed"].astype(int)
)

In [77]:
effort_threshold = analysis_df["customer_effort_score"].quantile(0.75)

analysis_df["high_effort"] = analysis_df["customer_effort_score"] > effort_threshold

In [78]:
priority_map = {
    "Low": 1,
    "Medium": 2,
    "High": 3,
    "Critical": 4
}

analysis_df["priority_score"] = analysis_df["ticket_priority"].map(priority_map)

In [79]:
analysis_df["description_length"] = analysis_df["ticket_description"].str.len()
analysis_df["resolution_length"] = analysis_df["resolution"].fillna("").str.len()

In [80]:
# Reduce noisy categories
top_types = analysis_df["ticket_type"].value_counts().nlargest(5).index

analysis_df["ticket_type_grouped"] = np.where(
    analysis_df["ticket_type"].isin(top_types),
    analysis_df["ticket_type"],
    "Other"
)

In [81]:
analysis_df[[
    "resolution_time_hours",
    "response_delay_hours",
    "customer_effort_score"
]].describe()

,resolution_time_hours,response_delay_hours,customer_effort_score
count,1404.000000,1404.000000,1404.000000
mean,7.577932,21244.815396,1.007793
std,5.596637,5043.694828,0.951852
min,0.000000,12435.154722,0.000000
25%,3.000000,16968.769722,0.300000
50%,6.341667,21366.096806,0.634167
75%,11.354167,25549.777014,1.385417
max,23.466667,29942.468889,3.346667


In [82]:
analysis_df.to_csv("../data/processed/cleaned_tickets.csv", index=False)

print("Clean dataset ready for analysis.")

Clean dataset ready for analysis.
